# RouterGuard: safe Kaggle runner

Upload this notebook to Kaggle, select **GPU T4 ×2** and turn **Internet ON**, then run cells from the top. Phone verification may be required for GPU access.

**Default Run All:** check the GPU → clone the code → install dependencies in a separate environment → test a real 8-bit CUDA operation → download OLMoE → run a tiny smoke experiment → export results. Full experiments are **off by default**.

- This is post-hoc calibration/pruning, not training from scratch. No API key is normally required for the public model/data.
- Only one GPU is used; two T4s do not become a single 32 GB GPU.
- The first download is roughly 15 GB, plus datasets/dependencies. Allow substantial disk space and download time. No fixed runtime is promised.
- Stop on the first failure. Do not run later experiment cells until it is fixed. The last export cell can still be run manually to download failure logs.
- This notebook reduces common setup/memory failures; it cannot guarantee Kaggle availability or compatibility with every future image.


## 1. Configuration — keep these defaults for the first run

After the smoke test succeeds, set `RUN_FULL = True`, rerun the configuration cell, then the full-run cell. Variants require a successful full run **in the current kernel**. Leave the other flags off until you need them.

Setup uses a virtual environment that **borrows Kaggle's existing CUDA PyTorch**. It creates the environment **without `ensurepip`**, then uses Kaggle's existing pip with `--python` to install into that environment rather than changing global packages. Re-running setup also repairs an environment left incomplete by an earlier `ensurepip` failure, without deleting it. It is not a completely isolated environment: unrelated Gradio/Diffusers conflict warnings can still appear. The checks below test what RouterGuard actually uses. You do not need to restart the kernel after this setup: every project command uses a fresh child Python process.


In [ ]:
from pathlib import Path

RUN_FULL = False
RUN_PROPORTIONAL = False
RUN_LAYER_BUDGET = False
ALLOW_LOW_DISK = False  # Only change after checking available disk yourself.

REPO_URL = 'https://github.com/harshalsapa7/nlpGA-.git'
REPO_BRANCH = 'master'
REPO_DIR = Path('/kaggle/working/nlpGA-')
RESULTS_DIR = Path('/kaggle/working/routerguard-results')
LOGS_DIR = Path('/kaggle/working/routerguard-logs')
# Keep model weights and the environment out of saved notebook outputs.
# These temporary files are NOT preserved across fresh Kaggle sessions.
CACHE_DIR = Path('/kaggle/temp/routerguard-hf')
ENV_DIR = Path('/kaggle/temp/routerguard-env')
PYTHON = ENV_DIR / 'bin/python'


In [ ]:
import datetime
import json
import os
import shutil
import subprocess
import sys

if not Path('/kaggle/working').is_dir():
    raise RuntimeError('This notebook is configured for Kaggle. Upload it there first.')

for directory in (RESULTS_DIR, LOGS_DIR, CACHE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

CHILD_ENV = os.environ.copy()
CHILD_ENV.update({
    'USE_TF': '0', 'USE_FLAX': '0', 'PYTHONUNBUFFERED': '1',
    'HF_HOME': str(CACHE_DIR),
    'HF_HUB_CACHE': str(CACHE_DIR / 'hub'),
    'HF_DATASETS_CACHE': str(CACHE_DIR / 'datasets'),
    'TOKENIZERS_PARALLELISM': 'false',
    'PIP_DISABLE_PIP_VERSION_CHECK': '1',
})
# Do not inject notebook paths into the project's child interpreter.
CHILD_ENV.pop('PYTHONPATH', None)
CHILD_ENV.pop('PYTHONHOME', None)
SETUP_OK = False
PREFLIGHT_OK = False
SMOKE_OK = False
FULL_OK = False

def run_checked(command, label, cwd=None):
    """Stream output, retain a log, stop on failure, and clean up on interrupt."""
    stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M%S-%f')
    log_path = LOGS_DIR / f'{stamp}_{label}.log'
    command = [str(part) for part in command]
    print(f'[{label}] Log: {log_path}', flush=True)
    with log_path.open('w', encoding='utf-8') as log:
        process = subprocess.Popen(
            command, cwd=cwd, env=CHILD_ENV, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace',
        )
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
        finally:
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=20)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
            process.stdout.close()
    if code != 0:
        raise RuntimeError(f'{label} failed (exit {code}). Stop here; inspect {log_path}.')


## 2. Check GPU access before installing anything

If this fails, enable **GPU T4 ×2** in Kaggle settings and restart the session if prompted. A CPU-only session cannot run this notebook's experiments. Do not try to fix GPU allocation by reinstalling CUDA or PyTorch.

The next cell clones `REPO_BRANCH` from `REPO_URL`, or fetches and fast-forwards an existing **clean checkout on that branch**. It stops rather than discarding local edits or divergent commits. Rerun that cell and preflight after a code fix; the printed revision identifies the code being tested. Updating GitHub does not change code already loaded in an older uploaded notebook.


In [ ]:
if not shutil.which('nvidia-smi'):
    raise RuntimeError('No NVIDIA tools found. Enable a Kaggle GPU before continuing.')
run_checked(['nvidia-smi'], 'nvidia-smi')
gpu_check = '''
import torch
print('PyTorch:', torch.__version__, 'CUDA build:', torch.version.cuda)
assert torch.cuda.is_available(), 'PyTorch cannot access CUDA. Check the Kaggle accelerator.'
print('GPU:', torch.cuda.get_device_name(0))
free, total = torch.cuda.mem_get_info(0)
print(f'GPU 0 memory: {free / 2**30:.1f} GiB free / {total / 2**30:.1f} GiB total')
assert total >= 14 * 2**30, 'Use a GPU with at least T4-class memory (16 GB).'
assert free >= 10 * 2**30, 'GPU memory is already occupied. Stop other GPU jobs first.'
'''
run_checked([sys.executable, '-c', gpu_check], 'base-pytorch-gpu')


In [ ]:
PREFLIGHT_OK = SMOKE_OK = FULL_OK = False
if not REPO_DIR.exists():
    run_checked(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, REPO_DIR], 'clone')
else:
    # Update only a clean checkout on the expected branch. Never reset or discard edits.
    origin = subprocess.check_output(
        ['git', '-C', str(REPO_DIR), 'remote', 'get-url', 'origin'], text=True,
    ).strip()
    if origin.rstrip('/').removesuffix('.git') != REPO_URL.removesuffix('.git'):
        raise RuntimeError(f'Existing checkout has another origin: {origin}. Check it manually.')
    status = subprocess.check_output(
        ['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True,
    ).strip()
    if status:
        raise RuntimeError('Checkout has local changes. Save/review them before updating:\n' + status)
    branch = subprocess.check_output(
        ['git', '-C', str(REPO_DIR), 'symbolic-ref', '--quiet', '--short', 'HEAD'], text=True,
    ).strip()
    if branch != REPO_BRANCH:
        raise RuntimeError(f'Expected branch {REPO_BRANCH}, found {branch}. Check it manually.')
    run_checked(['git', '-C', REPO_DIR, 'fetch', 'origin', REPO_BRANCH], 'fetch-code')
    run_checked(['git', '-C', REPO_DIR, 'merge', '--ff-only', 'FETCH_HEAD'], 'update-code')
    print('Checkout is current; only fast-forward updates were allowed.')

if not (REPO_DIR / 'routerguard/cli.py').is_file():
    raise RuntimeError('This checkout is missing the v2 CLI. Check the repository/branch.')
CODE_SHA = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True,
).strip()
print('Code revision:', CODE_SHA)


In [ ]:
SETUP_OK = PREFLIGHT_OK = SMOKE_OK = FULL_OK = False
# Kaggle images may fail inside ensurepip. Do not invoke it at all.
# Always reconfigure: a failed older attempt can leave bin/python present.
# No --clear and no deletion of existing environment files.
run_checked([
    sys.executable, '-m', 'venv', '--system-site-packages', '--without-pip', ENV_DIR,
], 'create-environment')
run_checked([
    PYTHON, '-c',
    'import pathlib, sys; '
    'assert pathlib.Path(sys.prefix).resolve() == pathlib.Path(sys.argv[1]).resolve(), '
    '"Wrong environment prefix; refusing to install"; '
    'assert sys.prefix != sys.base_prefix, "Expected a virtual environment"; '
    'print("Target environment:", sys.prefix)',
    ENV_DIR,
], 'verify-environment')
# --python explicitly targets the environment, even if it has no pip of its own.
# Requires pip >=22.3; Kaggle's pip version is checked by this command.
PIP_COMMAND = [sys.executable, '-m', 'pip', '--python', str(PYTHON)]
run_checked([*PIP_COMMAND, '--version'], 'pip-target')

# Prevent pip from replacing Kaggle's GPU-enabled PyTorch with another build.
torch_version = subprocess.check_output(
    [sys.executable, '-c', 'import torch; print(torch.__version__)'],
    text=True, env=CHILD_ENV,
).strip()
constraints = ENV_DIR / 'kaggle-torch-constraint.txt'
constraints.write_text(f'torch=={torch_version}\n', encoding='utf-8')
packages = [
    'transformers==4.48.3', 'huggingface-hub>=0.24,<1.0',
    'datasets>=2.19,<5', 'bitsandbytes>=0.43.3', 'accelerate>=0.34',
    'safetensors', 'pyyaml', 'matplotlib', 'sentencepiece',
]
run_checked([
    *PIP_COMMAND, 'install', '--upgrade-strategy', 'only-if-needed',
    '--constraint', constraints, *packages,
], 'install-dependencies')
SETUP_OK = True
print('Dependencies installed. Continue below; no kernel restart is needed.')


## 3. Preflight — versions, real 8-bit operation, disk space

The earlier `bitsandbytes` error must be resolved here **before** downloading the model. This also runs a complete tiny OLMoE pipeline on CUDA with synthetic tokens: calibration, similarity, pruning simulations, materialization and perplexity. It checks more than a model import or a calibration-only test, without downloading any weights. The tiny model is unquantized; the separate bitsandbytes operation checks actual int8 CUDA execution. TensorFlow is disabled for child processes because this project only uses PyTorch.

If this cell fails, download/share the `routerguard-logs` files rather than running the pipeline anyway. A pip resolver warning about unused Gradio/Diffusers packages is different from a failing import or CUDA operation.


In [ ]:
PREFLIGHT_OK = SMOKE_OK = FULL_OK = False
assert SETUP_OK, 'Run the dependency-install cell successfully first.'
preflight = '''
import importlib.metadata as md
import json
import torch
import bitsandbytes as bnb
import tempfile
from types import SimpleNamespace
from transformers import OlmoeConfig, OlmoeForCausalLM
from routerguard.balance import fit_biases
from routerguard.data import TokenSplit
from routerguard.pipeline import PipelineSpec, run_pipeline
from routerguard.env import assert_env
report = assert_env(strict=True)
print(json.dumps(report, indent=2))
for package in ('transformers', 'huggingface-hub', 'datasets', 'accelerate', 'bitsandbytes'):
    print(package, md.version(package))
layer = bnb.nn.Linear8bitLt(64, 64, bias=False, has_fp16_weights=False).to('cuda:0')
with torch.inference_mode():
    output = layer(torch.randn(2, 64, device='cuda:0', dtype=torch.float16))
torch.cuda.synchronize()
assert torch.isfinite(output).all().item(), '8-bit CUDA operation produced invalid values.'
# Exercise the complete tiny pipeline BEFORE the large download.
torch.manual_seed(0)
config = OlmoeConfig(
    vocab_size=128, hidden_size=64, intermediate_size=32, num_hidden_layers=2,
    num_attention_heads=4, num_key_value_heads=4, num_experts=8,
    num_experts_per_tok=3, max_position_embeddings=64,
)
tiny = OlmoeForCausalLM(config).to(device='cuda:0', dtype=torch.float16).eval()
biases, fit = fit_biases(
    tiny, torch.randint(0, 128, (2, 16)), batch_size=1, device='cuda:0', progress=False,
)
assert fit['updates'] == 2, 'Tiny calibration did not complete.'
assert all(b.device == next(tiny.parameters()).device and torch.isfinite(b).all().item()
           for b in biases.values())
splits = {
    'calib': TokenSplit('synthetic_calib', torch.randint(0, 128, (2, 16))),
    'eval': TokenSplit('synthetic_eval', torch.randint(0, 128, (1, 16))),
}
# Keep diagnostic metrics out of real experiment results; this is NOT evidence.
with tempfile.TemporaryDirectory(prefix='routerguard-preflight-') as root:
    state = run_pipeline(PipelineSpec(
        device='cuda:0', precision='fp16', tag='synthetic_preflight_NOT_PROTOCOL',
        loader=lambda: (tiny, None, SimpleNamespace(precision='fp16')),
        splits=splits, results_root=root, ratios=[0.25], seeds=[42],
        stats_batch_size=1, similarity_batch_size=1, ppl_batch_size=1,
        max_calib_tokens=32, max_eval_tokens=16, max_tokens_per_layer=32,
        with_refit=False, with_merge=False, do_mmlu=False, do_throughput=False,
        progress=False,
    ))
    assert state['baseline_ppl'] > 0 and state['pruned_ppl_balanced'] > 0
    assert state['checkpoint_bytes_saved'] > 0
print('PASS: environment, actual 8-bit CUDA operation, and complete tiny OLMoE CUDA pipeline.')
'''
run_checked([PYTHON, '-c', preflight], 'preflight', cwd=REPO_DIR)
run_checked([*PIP_COMMAND, 'freeze'], 'installed-packages', cwd=REPO_DIR)
for directory, minimum_gib in ((CACHE_DIR, 22), (RESULTS_DIR, 2)):
    free_gib = shutil.disk_usage(directory).free / 2**30
    print(f'{directory}: {free_gib:.1f} GiB free')
    if free_gib < minimum_gib and not ALLOW_LOW_DISK:
        raise RuntimeError(
            f'Conservative disk check: need {minimum_gib} GiB free at {directory}. '
            'Free space or review the cache location before continuing. '
            'ALLOW_LOW_DISK bypasses this estimate, not the need for storage.'
        )
PREFLIGHT_OK = True


In [ ]:
# Batch size 1 reduces activation memory. Keep all full-protocol sample counts unchanged.
BASE_SETTINGS = {
    'model': 'olmoe', 'precision': '8bit', 'device': 'cuda:0',
    'stats_batch_size': 1, 'ppl_batch_size': 1,
    'mmlu_batch_size': 1, 'similarity_batch_size': 1,
    'with_merge': False, 'results_root': str(RESULTS_DIR), 'progress': True,
}
# Explicitly shrink the DATA too: --fast alone does not cap every evaluation.
# These overrides are SMOKE ONLY, recorded as such, never research evidence.
SMOKE_SETTINGS = {
    'tag': 'kaggle_smoke_NOT_PROTOCOL', 'ratios': [0.25], 'seeds': [42],
    'with_refit': False, 'do_mmlu': False, 'do_throughput': False,
    'max_calib_tokens': 1024, 'max_eval_tokens': 512, 'max_tokens_per_layer': 1024,
    'overrides': {
        'calibration': {'n_seqs': 4, 'seq_len': 256},
        'eval': {'n_seqs': 2, 'seq_len': 256},
    },
}

def run_experiment(settings, label):
    assert PREFLIGHT_OK, 'Complete the preflight successfully first.'
    config = {**BASE_SETTINGS, **settings}
    manifest = {
        'code_sha': CODE_SHA, 'settings': config,
        'started_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
        'smoke_only': label == 'smoke',
    }
    stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M%S-%f')
    (LOGS_DIR / f'{stamp}_{label}_settings.json').write_text(
        json.dumps(manifest, indent=2), encoding='utf-8',
    )
    # A fresh process releases model/GPU memory when it exits and cannot reuse
    # stale notebook imports. Return code is checked: a traceback is not success.
    program = '''
import json
import sys
from pathlib import Path
from routerguard.pipeline import PipelineSpec, run_pipeline
manifest = json.loads(sys.argv[1])
state = run_pipeline(PipelineSpec(**manifest['settings']))
path = Path(state['run_dir']) / 'notebook_settings.json'
path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('SUCCESS:', state['run_dir'])
'''
    run_checked([PYTHON, '-c', program, json.dumps(manifest)], label, cwd=REPO_DIR)


## 4. Tiny end-to-end smoke test

This **still downloads and loads the full OLMoE checkpoint**. Only its calibration/evaluation workload is small. It checks the pipeline, not scientific quality. Do not put its numbers in the final results table. It skips MMLU, throughput and post-pruning refit, and tests only 25% pruning with one random seed.

The first download may be slow. Logs are streamed and saved. If the session is interrupted, download the logs/results before stopping it; this notebook does not promise automatic whole-pipeline resume.


In [ ]:
SMOKE_OK = FULL_OK = False
run_experiment(SMOKE_SETTINGS, 'smoke')
SMOKE_OK = True
print('Smoke passed. Full experiments below remain off until you opt in.')


## 5. Optional full registered experiment

Set `RUN_FULL = True` in section 1, rerun **only the configuration cell**, then run the next cell. It keeps the protocol's sample counts, pruning ratios and seeds, and enables MMLU, throughput and bias refit. Batch sizes remain 1; this is saved in the run settings. Throughput comparisons must use matching batch sizes/hardware.

Expect substantially more work than the smoke test; check your remaining Kaggle quota. Download/save the results after each completed run. If MMLU is unavailable, the pipeline records a note and continues with perplexity: report that omission rather than claiming MMLU was evaluated. Read any gate failures and extension warnings honestly; a controlled negative result is valid.


In [ ]:
if RUN_FULL:
    FULL_OK = False
    assert SMOKE_OK, 'Run the smoke test successfully in this kernel first.'
    run_experiment({'tag': 'kaggle_full'}, 'full')
    FULL_OK = True
else:
    print('Skipped full run. Set RUN_FULL=True in section 1 when ready.')


## 6. Optional v2 variants — off by default

Each option is another full experiment, not a cheap extra evaluation. Enable them separately only after the main full run completes. No merge arm or Qwen run is launched here; both add complexity and are outside this notebook's safe T4 path.


In [ ]:
if RUN_PROPORTIONAL:
    assert FULL_OK, 'Complete the main full run in this kernel first.'
    run_experiment({'tag': 'kaggle_proportional', 'update_mode': 'proportional'}, 'proportional')
else:
    print('Skipped proportional variant.')


In [ ]:
if RUN_LAYER_BUDGET:
    assert FULL_OK, 'Complete the main full run in this kernel first.'
    run_experiment({'tag': 'kaggle_layer_budget', 'layer_wise_budget': True}, 'layer-budget')
else:
    print('Skipped layer-budget variant.')


## 7. Inspect and download results (also works after a failure)

Run this cell manually after any failure to package the logs collected so far. Summaries live at `routerguard-results/<run_id>/metrics/summary.json`. An interrupted run may contain partial metrics but no summary.

Download **routerguard-kaggle-results.zip** using the link or Kaggle's file/output panel **before ending the session**. It contains results, resolved configs, revision records, package versions and logs, but not model weights or the environment. To persist outputs in Kaggle, save a notebook version with current outputs if available. **Save & Run All** launches a fresh execution and repeats whichever experiments are enabled; it is not a free save of an existing run.


In [ ]:
import csv
import zipfile
from IPython.display import FileLink, display

rows = []
for summary_path in sorted(RESULTS_DIR.glob('*/metrics/summary.json')):
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    run_dir = summary_path.parent.parent
    metadata_path = run_dir / 'notebook_settings.json'
    metadata = json.loads(metadata_path.read_text()) if metadata_path.exists() else {}
    row = {
        'run': run_dir.name,
        'smoke_only': metadata.get('smoke_only', 'unknown'),
        'baseline_ppl': summary.get('baseline_ppl'),
        'pruned_ppl_balanced': summary.get('pruned_ppl_balanced'),
        'refit_ppl': summary.get('refit_ppl'),
        'checkpoint_bytes_saved': summary.get('checkpoint_bytes_saved'),
        'elapsed_s': summary.get('elapsed_s'),
    }
    rows.append(row)
    print(json.dumps(row, indent=2))
    print('Run notes:', summary.get('notes', []))
    for metric_name in ('gates', 'balance'):
        metric_path = summary_path.parent / f'{metric_name}.json'
        if metric_path.exists():
            print(metric_name + ':', metric_path.read_text(encoding='utf-8'))

if rows:
    with (RESULTS_DIR / 'summary.csv').open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
else:
    print('No completed run yet. The archive will still include logs and partial results.')

archive_path = Path('/kaggle/working/routerguard-kaggle-results.zip')
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for root in (RESULTS_DIR, LOGS_DIR):
        for path in sorted(root.rglob('*')):
            if path.is_file() and not path.is_symlink():
                archive.write(path, arcname=str(Path(root.name) / path.relative_to(root)))
print(f'Archive: {archive_path} ({archive_path.stat().st_size / 2**20:.1f} MiB)')
display(FileLink(os.path.relpath(archive_path, Path.cwd())))


## If something fails

| Failure | What to do |
|---|---|
| No GPU / CUDA unavailable | Enable the Kaggle GPU. If `nvidia-smi` works but PyTorch fails, share the base-PyTorch log; do not reinstall CUDA blindly. |
| An older notebook failed at `ensurepip` | Use this updated setup cell. It creates/reconfigures the environment with `--without-pip` and uses Kaggle's pip to target it explicitly; no environment deletion is needed. |
| `pip-target` reports `no such option: --python` | The host pip is older than 22.3. Share that log and the base pip version before changing the system environment. |
| pip cannot install a compatible package / missing wheel | Share `install-dependencies.log` and the notebook's Python version. This environment may need a different Kaggle image or a separate supported Python; do not remove the project pins. |
| `bitsandbytes` import/kernel error | Share the preflight log. The notebook tests an actual operation before the large download. Do not continue to the pipeline. |
| `fit_biases` reports tensors on `cuda:0` and `cpu` | Use the updated notebook and rerun its clone/update cell, then preflight and smoke. Older clone cells do not fetch fixes. This version transfers recorder counts to the accumulator's device and tests tiny CUDA calibration before downloading weights. |
| Disk check fails | Review space at the printed locations. Remove only your own unneeded files or choose another cache location, then rerun setup/preflight. Model downloads are not free of disk cost. |
| Download timeout / rate limit | Confirm Internet is on, wait if rate-limited, then retry the failed cell. Cache files are reused within the same session; do not delete them as a first step. |
| CUDA out of memory | Stop other GPU jobs. This notebook already uses batch size 1. Export logs; do not silently switch precision/sample counts and call the numbers the same experiment. |
| Session interrupted | Restore setup in the new session, rerun checks and smoke. Downloaded weights in temporary storage and unsaved results may be gone. No automatic full-run resume is promised. |

GPU allocation can consume quota while idle. After downloading results and saving any needed outputs, stop the Kaggle session.
